# LR: chain height and lateral displacement of Lo- and Ld-classified chains (table)

For every cholesterol fraction χ_C, the sn-1 and sn-2 **height** (|z| of the C2 → C18 vector) and **lateral displacement**
(xy length of the same vector), split by the LR label (Lo / Ld). The table gives the **mean** in nm (no uncertainties).

- PBC correction: before the calculation, every DOPC is made whole (minimum image to one of its own atoms, box size
  of each frame). Some molecules, about 25 % in `dopc_pure`, are split across the box in the saved positions. This is
  done in memory for this notebook only; the saved positions are not changed.
- Height and lateral displacement are time-averaged over the same 7-pt window as the classifier, so every value has
  exactly one label (`prediction_labels` from `LR_save_classifier_data`).
- Mixed systems (χ_C = 0.0625 … 0.35): both leaflets pooled.
- Endpoints, pure DOPC (χ_C = 0) and DOPC/CHOL 60/40 (χ_C = 0.40): **upper leaflet only**, because their lower leaflet
  was the training leaflet.
- χ_C = 0.25: the value is from `dopc_75`. Its uncertainty, the SD (ddof = 1) of the class means of the three
  χ_C = 0.25 runs (`dopc_75`, `dopc_75_rep1`, `dopc_75_rep2`), is printed in the last cell, not in the table.
- A second table gives the number of chain-frames behind every value (small numbers, e.g. Lo in pure DOPC, are less reliable).

Output: `heavy_data_NO_GIT/results/chain_height_lat_d/log_reg_7pt_height_lat_d_lo_ld_table.csv` (formatted table),
`..._long.csv` (all numbers) and `..._counts.csv`.

In [ ]:
import sys
sys.path.insert(0, "..")          # code_libraries is in the repository root; data in ../heavy_data_NO_GIT

import os
import gc
import numpy as np
import pandas as pd

from code_libraries.acyl_chain_preprocessing import chain_height_df
from code_libraries.op_by_chain import convolve_ops
from code_libraries.predict_acyl_chains_NN_by_chain import clf_lo_ld_by_chain

start_frame, stop_frame = 0, 4000        # same range as the classification
sys_type = "sym"
window = 7                               # window used in LR_save_classifier_data
red_type = "ft_last"
model_type, model_label = "log_reg", "LR"
validation_frames = 3

data_array_dir = os.path.join("../", "heavy_data_NO_GIT/data_arrays_by_chain_logreg")
output_dir = os.path.join("../", "heavy_data_NO_GIT/results/chain_height_lat_d")
os.makedirs(output_dir, exist_ok=True)

# (system, chi_C, leaflets used)
systems = [
    ("dopc_pure", 0.0,    ["upper"]),            # endpoint: upper (unseen) leaflet only
    ("dopc_94",   0.0625, ["lower", "upper"]),
    ("dopc_90",   0.10,   ["lower", "upper"]),
    ("dopc_85",   0.15,   ["lower", "upper"]),
    ("dopc_80",   0.20,   ["lower", "upper"]),
    ("dopc_75",   0.25,   ["lower", "upper"]),
    ("dopc_70",   0.30,   ["lower", "upper"]),
    ("dopc_65",   0.35,   ["lower", "upper"]),
    ("dopc_60",   0.40,   ["upper"]),            # endpoint: upper (unseen) leaflet only
]
replicas = [("dopc_75_rep1", 0.25, ["lower", "upper"]),
            ("dopc_75_rep2", 0.25, ["lower", "upper"])]   # with dopc_75: error bar at chi_C = 0.25

chains = ["sn1", "sn2"]
classes = ["Lo", "Ld"]
quantities = {"height": 0, "lateral": 1}         # column in the (frames, lipids, 2) array

In [ ]:
# Constants mirror chain_height_df(); the check below confirms they still match.
sn1_indices = [41, 42, 43, *range(91, 138)]
sn2_indices = [32, 33, 34, *range(44, 91)]
chain_masses = np.array(
    [
        12., 1., 1., 12., 1., 1., 12., 1., 1., 12., 1., 1., 12.,
        1., 1., 12., 1., 1., 12., 1., 1., 12., 1., 12., 1., 12.,
        1., 1., 12., 1., 1., 12., 1., 1., 12., 1., 1., 12., 1.,
        1., 12., 1., 1., 12., 1., 1., 12., 1., 1., 1.,
    ],
    dtype=float,
)


def fast_chain_height_array(dopc_leaflet, chain):
    """(frames, lipids, 2): [height, lateral displacement] of the C2 -> C18 vector, as in chain_height_df."""
    chain_indices = sn1_indices if chain == "sn1" else sn2_indices
    chain_positions = dopc_leaflet[:, :, chain_indices, :]

    chain_com = np.sum(chain_positions * chain_masses[None, None, :, None], axis=2) / chain_masses.sum()
    shifted_chain_positions = chain_positions - chain_com[:, :, None, :]

    cstart = shifted_chain_positions[:, :, 0, :]
    cend = shifted_chain_positions[:, :, -4, :]

    z_height = np.abs(cstart[:, :, 2] - cend[:, :, 2])
    xy_radius = np.linalg.norm(cstart[:, :, :2] - cend[:, :, :2], axis=2)

    return np.stack([z_height, xy_radius], axis=2)


def check_against_chain_height_df(dopc_leaflet, chain):
    """The fast array must equal chain_height_df on the first few frames (rows lipid by lipid)."""
    n_check = min(validation_frames, dopc_leaflet.shape[0])
    check_df = chain_height_df(dopc_leaflet[:n_check], n_check, chain, iqr_filter=False)
    check_fast = fast_chain_height_array(dopc_leaflet[:n_check], chain).transpose(1, 0, 2).reshape(-1, 2)
    np.testing.assert_allclose(check_fast, check_df[["z_height", "xy_radius"]].to_numpy(dtype=float),
                               rtol=1e-12, atol=1e-12)


def make_whole(dopc_leaflet, box, ref_atom=32, chunk=250):
    """PBC correction for this calculation only (the saved positions are not changed).
    Moves every atom of a DOPC to the periodic image closest to a reference atom of the same molecule
    (minimum image; atom 32 sits next to the glycerol). Needed because some trajectories, mainly dopc_pure,
    contain molecules split across the box. Molecules that are already whole are not changed.
    dopc_leaflet: (frames, lipids, 138, 3) nm, modified in place; box: (frames, 3) nm."""
    n_moved = 0
    for a in range(0, dopc_leaflet.shape[0], chunk):
        p = dopc_leaflet[a:a + chunk]
        L = box[a:a + chunk][:, None, None, :]
        shift = -L * np.round((p - p[:, :, ref_atom:ref_atom + 1, :]) / L)
        n_moved += int(np.any(shift != 0, axis=-1).any(axis=-1).sum())
        p += shift
    return n_moved                                                   # number of (lipid, frame) pairs corrected


def classified_values(sys_name, leaflets_used):
    """{(chain, class): (n, 2) array of [height, lateral]} over all chain-frames of the chosen leaflets."""
    dopc_lower, dopc_upper = clf_lo_ld_by_chain(sys_name, sys_type, "raw", model_type, "sn1", 3,
                                                start_frame, stop_frame).bilayer_all_dopc_data()
    dopc_by_leaflet = {"lower": np.array(dopc_lower, dtype=float), "upper": np.array(dopc_upper, dtype=float)}
    del dopc_lower, dopc_upper
    # PBC correction (this calculation only): box size of every frame, saved by LR_save_classifier_data
    box = np.load(os.path.join(data_array_dir, f"{sys_name}_sn1_{red_type}_{window}pt_box_dim.npy"), allow_pickle=True)
    for leaflet in leaflets_used:
        assert box.shape == (dopc_by_leaflet[leaflet].shape[0], 3), (box.shape, dopc_by_leaflet[leaflet].shape)
        n_fixed = make_whole(dopc_by_leaflet[leaflet], box)
        n_total = dopc_by_leaflet[leaflet].shape[0] * dopc_by_leaflet[leaflet].shape[1]
        print(f"   {leaflet} leaflet: {n_fixed} of {n_total} lipid-frames were split across the box and made whole")
    out = {}
    for chain in chains:
        labels = np.load(os.path.join(data_array_dir, f"{sys_name}_{chain}_{red_type}_{window}pt_prediction_labels.npy"),
                         allow_pickle=True)
        labels_by_leaflet = {"lower": np.asarray(labels[0], dtype=bool), "upper": np.asarray(labels[1], dtype=bool)}
        values = {"Lo": [], "Ld": []}
        for leaflet in leaflets_used:
            dopc_leaflet = dopc_by_leaflet[leaflet]
            check_against_chain_height_df(dopc_leaflet, chain)
            averaged = convolve_ops(fast_chain_height_array(dopc_leaflet, chain), window)   # (frames - window + 1, lipids, 2)
            lab = labels_by_leaflet[leaflet]
            if averaged.shape[:2] != lab.shape:
                raise ValueError(f"{sys_name} {chain} {leaflet}: features {averaged.shape[:2]} vs labels {lab.shape}")
            values["Lo"].append(averaged[lab])
            values["Ld"].append(averaged[~lab])
        for cls in classes:
            out[(chain, cls)] = np.concatenate(values[cls]) if values[cls] else np.empty((0, 2))
    del dopc_by_leaflet
    gc.collect()
    return out

In [ ]:
# mean, SD and number of chain-frames for every system, chain, class and quantity (slow: loads each system once)
rows = []
for sys_name, chi, leaflets_used in systems + replicas:
    print(f"{sys_name} (chi_C = {chi}, leaflets: {', '.join(leaflets_used)})", flush=True)
    vals = classified_values(sys_name, leaflets_used)
    for (chain, cls), v in vals.items():
        for q, j in quantities.items():
            x = v[:, j]
            rows.append({"system": sys_name, "chi_C": chi, "chain": chain, "class": cls, "quantity": q,
                         "mean": x.mean() if x.size else np.nan, "sd": x.std() if x.size else np.nan,
                         "n_chain_frames": int(x.size)})
    del vals
    gc.collect()

long = pd.DataFrame(rows)
long.to_csv(os.path.join(output_dir, f"{model_type}_{window}pt_height_lat_d_lo_ld_long.csv"), index=False)

In [ ]:
# table: means only (nm); the chi_C = 0.25 uncertainty is printed in the last cell
table_vals = long[long.system.isin([s[0] for s in systems])].copy()

row_name = {"dopc_pure": "0 (pure DOPC)", "dopc_60": "0.40"}
order = [s[0] for s in systems]
columns = [(f"{ch[:2]}-{ch[2]} {q} (nm)", cls) for ch in chains for q in quantities for cls in classes]


def build(value):
    t = pd.DataFrame(index=[row_name.get(s, f"{c:g}") for s, c, _ in systems],
                     columns=pd.MultiIndex.from_tuples(columns))
    t.index.name = "Xc"
    for _, r in table_vals.iterrows():
        name = row_name.get(r.system, f"{r.chi_C:g}")
        t.loc[name, (f"{r.chain[:2]}-{r.chain[2]} {r.quantity} (nm)", r["class"])] = value(r)
    return t


table = build(lambda r: f"{r['mean']:.3f}" if np.isfinite(r["mean"]) else "–")
counts = build(lambda r: int(r["n_chain_frames"]))

table.to_csv(os.path.join(output_dir, f"{model_type}_{window}pt_height_lat_d_lo_ld_table.csv"))
counts.to_csv(os.path.join(output_dir, f"{model_type}_{window}pt_height_lat_d_lo_ld_counts.csv"))
print(f"{model_label} {window}pt: mean (nm)")
table

In [ ]:
# number of chain-frames behind each value
counts

In [ ]:
# uncertainty at chi_C = 0.25: SD (ddof = 1) of the class means of dopc_75, dopc_75_rep1, dopc_75_rep2
runs_025 = ["dopc_75"] + [r[0] for r in replicas]
rep = long[long.system.isin(runs_025)]
rep_means = rep.pivot_table(index=["chain", "quantity", "class"], columns="system", values="mean")[runs_025]
rep_means["replica SD"] = rep_means[runs_025].std(axis=1, ddof=1)

print(f"{model_label} {window}pt, chi_C = 0.25: class means of the three runs and their SD (nm)\n")
print(rep_means.round(4).to_string(), "\n")
for (chain, q, cls), r in rep_means.iterrows():
    print(f"{chain[:2]}-{chain[2]} {q:8s} {cls}: {r['dopc_75']:.3f} ± {r['replica SD']:.3f} nm")